In [ ]:
#!/usr/bin/env python3
"""
SPL.com.sa — Saudi Pro League API Scraper  (v2 — direct JSON API, no Selenium)
================================================================================
Calls the PulseLive API that powers spl.com.sa directly:
    https://api.saudi-pro-league.pulselive.com/football/...

This is MUCH simpler/faster than Selenium scraping — pure HTTP + JSON.

Sections scraped per season:
  • Fixtures/Results — every match: date, teams, score, venue, attendance, outcome
  • Standings        — Pos, Team, P, W, D, L, GF, GA, GD, Pts  (built from fixtures
                        if a dedicated standings endpoint isn't available)

All seasons → spl_data/master.csv
Per-season CSVs also saved.

INSTALL:
    pip install requests pandas colorama

RUN:
    python spl_api_scraper.py
"""

import os, sys, json, time
from datetime import datetime

try:
    import requests
except ImportError:
    sys.exit("pip install requests")

try:
    import pandas as pd
except ImportError:
    sys.exit("pip install pandas")

try:
    from colorama import init, Fore, Style
    init(autoreset=True)
except ImportError:
    class Fore:
        GREEN=CYAN=RED=YELLOW=WHITE=MAGENTA=RESET=""
    class Style:
        BRIGHT=RESET_ALL=""


# ══════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════
API = "https://api.saudi-pro-league.pulselive.com/football"
COMPETITION_ID = 215   # Saudi Arabian League (from sample response)

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0.0.0 Safari/537.36"),
    "Origin":  "https://www.spl.com.sa",
    "Referer": "https://www.spl.com.sa/",
    "Accept":  "application/json",
}

# Confirmed compSeason IDs for every Saudi Pro League season back to 2011/12
KNOWN_COMP_SEASONS = {
    "2025/26": 859,
    "2024/25": 858,
    "2023/24": 843,
    "2022/23": 844,
    "2021/22": 857,
    "2020/21": 856,
    "2019/20": 854,
    "2018/19": 853,
    "2017/18": 852,
    "2016/17": 851,
    "2015/16": 850,
    "2014/15": 848,
    "2013/14": 847,
    "2012/13": 846,
    "2011/12": 845,
}

# Which seasons to actually scrape this run — edit as needed.
SEASONS_TO_SCRAPE = list(KNOWN_COMP_SEASONS.keys())

OUT = "spl_data"
os.makedirs(OUT, exist_ok=True)
NOW = datetime.now().strftime("%Y-%m-%d %H:%M")


# ══════════════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════════════
def banner():
    print(f"""
{Fore.GREEN}{Style.BRIGHT}╔══════════════════════════════════════════════════════╗
║   ⚽  SPL.com.sa — Direct API Scraper (v2)          ║
║      No Selenium · Pure JSON · Master CSV           ║
╚══════════════════════════════════════════════════════╝{Style.RESET_ALL}
""")

def section(title):
    print(f"\n{Fore.CYAN}{Style.BRIGHT}{'─'*58}\n  {title}\n{'─'*58}{Style.RESET_ALL}")

def ok(m):   print(f"  {Fore.GREEN}✓{Style.RESET_ALL}  {m}")
def warn(m): print(f"  {Fore.YELLOW}⚠{Style.RESET_ALL}  {m}")
def err(m):  print(f"  {Fore.RED}✗{Style.RESET_ALL}  {m}")
def info(m): print(f"  {Fore.CYAN}→{Style.RESET_ALL}  {m}")


# ══════════════════════════════════════════════════════
#  API HELPERS
# ══════════════════════════════════════════════════════
def api_get(path: str, params: dict, debug_name: str = "") -> dict | None:
    url = f"{API}/{path}"
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=20)
        if r.status_code == 200:
            return r.json()
        warn(f"{r.status_code} from {url}  params={params}")
        if debug_name:
            dbg = os.path.join(OUT, f"debug_{debug_name}_{r.status_code}.txt")
            with open(dbg, "w", encoding="utf-8") as f:
                f.write(f"URL: {r.url}\nStatus: {r.status_code}\n\n{r.text[:2000]}")
            warn(f"Saved {dbg}")
    except Exception as e:
        err(f"Request error: {e}")
    return None






# ══════════════════════════════════════════════════════
#  FIXTURES / RESULTS
# ══════════════════════════════════════════════════════
def get_all_fixtures(comp_season_id: int, season_label: str) -> list[dict]:
    """
    Page through GET /fixtures?compSeasons=<id>&...  to get every match.
    No gameweek filter = should return matches across the whole season,
    paginated by pageSize.
    """
    all_matches = []
    page = 0
    page_size = 40

    while True:
        data = api_get("fixtures", {
            "compSeasons": comp_season_id,
            "comps":       COMPETITION_ID,
            "page":        page,
            "pageSize":    page_size,
            "sort":        "asc",
            "statuses":    "C,U,L",   # Completed, Upcoming, Live — get everything
            "provisional": "false",
        }, debug_name=f"fixtures_{season_label.replace('/','-')}")

        if not data:
            break

        content = data.get("content", [])
        if not content:
            break

        all_matches.extend(content)

        page_info = data.get("pageInfo", {})
        num_pages = page_info.get("numPages", 1)
        page += 1
        if page >= num_pages:
            break

        time.sleep(0.3)

    return all_matches


def parse_fixture(m: dict, season_label: str) -> dict:
    """Flatten one fixture/result JSON object into a flat row."""
    teams = m.get("teams", [{}, {}])
    home  = teams[0] if len(teams) > 0 else {}
    away  = teams[1] if len(teams) > 1 else {}

    home_team = home.get("team", {})
    away_team = away.get("team", {})

    home_name = home_team.get("name") or home_team.get("club", {}).get("name", "")
    away_name = away_team.get("name") or away_team.get("club", {}).get("name", "")

    home_score = home.get("score")
    away_score = away.get("score")

    gw = m.get("gameweek", {})
    kickoff = m.get("kickoff", {})
    ground  = m.get("ground", {})

    status  = m.get("status", "")    # C=completed, U=upcoming, L=live
    outcome = m.get("outcome", "")   # H/A/D

    row = {
        "section":     "results" if status == "C" else "fixtures",
        "season":      season_label,
        "gameweek":    gw.get("gameweek"),
        "match_id":    m.get("id"),
        "date_label":  kickoff.get("label", ""),
        "kickoff_ms":  kickoff.get("millis"),
        "home_team":   home_name,
        "away_team":   away_name,
        "home_goals":  home_score,
        "away_goals":  away_score,
        "status":      status,
        "outcome":     outcome,         # H / A / D
        "attendance":  m.get("attendance"),
        "venue":       ground.get("name", ""),
        "city":        ground.get("city", ""),
        "scraped_at":  NOW,
    }

    if home_score is not None and away_score is not None:
        try:
            hg, ag = int(home_score), int(away_score)
            row["home_goals"]  = hg
            row["away_goals"]  = ag
            row["score"]       = f"{hg}-{ag}"
            row["total_goals"] = hg + ag
            row["result"]      = "H" if hg > ag else "A" if ag > hg else "D"
        except (ValueError, TypeError):
            pass

    return row


# ══════════════════════════════════════════════════════
#  STANDINGS — built from fixtures
#  (avoids needing a separate, possibly-different standings endpoint)
# ══════════════════════════════════════════════════════
def build_standings_from_fixtures(fixtures: list[dict], season_label: str) -> list[dict]:
    """Compute a full league table from completed match results."""
    teams = {}

    def ensure(team):
        if team not in teams:
            teams[team] = {
                "team": team, "played": 0, "won": 0, "drawn": 0, "lost": 0,
                "gf": 0, "ga": 0, "form": [],
            }
        return teams[team]

    # Sort by kickoff time so "form" reflects chronological order
    completed = [f for f in fixtures if f["section"] == "results"
                  and f.get("home_goals") is not None]
    completed.sort(key=lambda f: f.get("kickoff_ms") or 0)

    for f in completed:
        h, a = f["home_team"], f["away_team"]
        hg, ag = f["home_goals"], f["away_goals"]
        th, ta = ensure(h), ensure(a)

        th["played"] += 1; ta["played"] += 1
        th["gf"] += hg;    th["ga"] += ag
        ta["gf"] += ag;    ta["ga"] += hg

        if hg > ag:
            th["won"] += 1; ta["lost"] += 1
            th["form"].append("W"); ta["form"].append("L")
        elif ag > hg:
            ta["won"] += 1; th["lost"] += 1
            th["form"].append("L"); ta["form"].append("W")
        else:
            th["drawn"] += 1; ta["drawn"] += 1
            th["form"].append("D"); ta["form"].append("D")

    rows = []
    for team, t in teams.items():
        played = t["played"] or 1
        points = t["won"] * 3 + t["drawn"]
        gd = t["gf"] - t["ga"]
        rows.append({
            "section":       "standings",
            "season":        season_label,
            "team":          team,
            "played":        t["played"],
            "won":           t["won"],
            "drawn":         t["drawn"],
            "lost":          t["lost"],
            "goals_for":     t["gf"],
            "goals_against": t["ga"],
            "goal_diff":     gd,
            "points":        points,
            "ppg":           round(points / played, 2),
            "win_pct":       round(t["won"] / played * 100, 1),
            "avg_scored":    round(t["gf"] / played, 2),
            "avg_conceded":  round(t["ga"] / played, 2),
            "form_last5":    "".join(t["form"][-5:]),
            "scraped_at":    NOW,
        })

    # Sort by points desc, then GD desc, then GF desc — standard tiebreakers
    rows.sort(key=lambda r: (-r["points"], -r["goal_diff"], -r["goals_for"]))
    for i, r in enumerate(rows, 1):
        r["pos"] = i

    # Reorder so pos is first
    ordered = []
    for r in rows:
        ordered.append({
            "section": r["section"], "season": r["season"], "pos": r["pos"],
            "team": r["team"], "played": r["played"], "won": r["won"],
            "drawn": r["drawn"], "lost": r["lost"],
            "goals_for": r["goals_for"], "goals_against": r["goals_against"],
            "goal_diff": r["goal_diff"], "points": r["points"], "ppg": r["ppg"],
            "win_pct": r["win_pct"], "avg_scored": r["avg_scored"],
            "avg_conceded": r["avg_conceded"], "form_last5": r["form_last5"],
            "scraped_at": r["scraped_at"],
        })
    return ordered


# ══════════════════════════════════════════════════════
#  DISPLAY
# ══════════════════════════════════════════════════════
def print_standings(data):
    print(f"\n  {'#':>3}  {'Team':<22} {'P':>3} {'W':>3} {'D':>3} {'L':>3}"
          f" {'GF':>4} {'GA':>4} {'GD':>5} {'Pts':>4}  {'Form':<6}")
    print(f"  {'─'*72}")
    for r in data:
        gd = f"+{r['goal_diff']}" if r["goal_diff"] > 0 else str(r["goal_diff"])
        print(f"  {r['pos']:>3}. {r['team']:<22}"
              f" {r['played']:>3} {r['won']:>3} {r['drawn']:>3} {r['lost']:>3}"
              f" {r['goals_for']:>4} {r['goals_against']:>4} {gd:>5}"
              f" {r['points']:>4}  {r['form_last5']:<6}")


def print_matches(data, label, n=10):
    if not data:
        return
    print(f"\n  {label} ({len(data)} total):")
    print(f"  {'GW':>3} {'Date':<28} {'Home':<16} {'Score':>7}  {'Away':<16} {'Status'}")
    print(f"  {'─'*82}")
    for r in data[:n]:
        sc = r.get("score", "vs")
        gw = r.get("gameweek")
        gw = int(gw) if gw is not None else "-"
        print(f"  {gw!s:>3} {r['date_label']:<28} {r['home_team']:<16}"
              f" {sc:>7}  {r['away_team']:<16} {r['status']}")
    if len(data) > n:
        print(f"  … +{len(data)-n} more rows")


# ══════════════════════════════════════════════════════
#  SAVE
# ══════════════════════════════════════════════════════
def save(data: list[dict], filename: str, all_frames: list):
    if not data:
        warn(f"No data — skipping {filename}")
        return
    df   = pd.DataFrame(data)
    path = os.path.join(OUT, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    ok(f"Saved {filename}  ({len(df)} rows)")
    all_frames.append(df)


# ══════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════
def main():
    banner()

    comp_seasons = {label: KNOWN_COMP_SEASONS[label]
                     for label in SEASONS_TO_SCRAPE
                     if label in KNOWN_COMP_SEASONS}

    missing = [l for l in SEASONS_TO_SCRAPE if l not in KNOWN_COMP_SEASONS]
    for m in missing:
        warn(f"Season '{m}' not in KNOWN_COMP_SEASONS — skipping.")

    if not comp_seasons:
        err("No valid seasons to scrape. Check SEASONS_TO_SCRAPE / KNOWN_COMP_SEASONS.")
        return

    print(f"\n  Seasons to scrape: {list(comp_seasons.items())}\n")

    all_frames = []

    for label, comp_id in comp_seasons.items():
        section(f"SEASON {label}  (compSeason={comp_id})")

        fixtures_raw = get_all_fixtures(comp_id, label)
        if not fixtures_raw:
            err(f"No fixtures returned for {label}.")
            continue

        ok(f"Fetched {len(fixtures_raw)} raw fixture records.")

        rows = [parse_fixture(m, label) for m in fixtures_raw]
        results  = [r for r in rows if r["section"] == "results"]
        fixtures = [r for r in rows if r["section"] == "fixtures"]

        short = label.replace("/", "-")
        save(results,  f"results_{short}.csv",  all_frames)
        save(fixtures, f"fixtures_{short}.csv", all_frames)
        print_matches(results, "Results")
        print_matches(fixtures, "Fixtures")

        standings = build_standings_from_fixtures(rows, label)
        save(standings, f"standings_{short}.csv", all_frames)
        print_standings(standings)

    # ── MASTER CSV ─────────────────────────────────
    if all_frames:
        master = pd.concat(all_frames, ignore_index=True, sort=False)
        master_path = os.path.join(OUT, "master.csv")
        master.to_csv(master_path, index=False, encoding="utf-8-sig")
        print(f"\n{Fore.GREEN}{Style.BRIGHT}")
        print(f"  ╔══════════════════════════════════════════════╗")
        print(f"  ║  MASTER CSV SAVED                           ║")
        print(f"  ║  {master_path:<44}║")
        print(f"  ║  {len(master):,} total rows · {len(master.columns)} columns            ║")
        print(f"  ╚══════════════════════════════════════════════╝")
        print(f"{Style.RESET_ALL}")
        print(f"  Sections: {sorted(master['section'].dropna().unique().tolist())}")
        print(f"  Seasons:  {sorted(master['season'].dropna().unique().tolist())}")
    else:
        err("Nothing scraped — master.csv not created.")

    # ── FILE SUMMARY ─────────────────────────────────
    print(f"\n  {'File':<32} {'Rows':>6}")
    print(f"  {'─'*40}")
    for fname in sorted(f for f in os.listdir(OUT) if f.endswith(".csv")):
        path = os.path.join(OUT, fname)
        n = sum(1 for _ in open(path, encoding="utf-8-sig")) - 1
        print(f"  {Fore.GREEN}✓{Style.RESET_ALL}  {fname:<30} {n:>6}")

    print(f"\n  All files saved to: {Fore.CYAN}./{OUT}/{Style.RESET_ALL}\n")


if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════╗
║   ⚽  Soccerway — Saudi Pro League Master Scraper   ║
║      Multi-season data scraper                      ║
║      All data → soccerway_data/master.csv           ║
╚══════════════════════════════════════════════════════╝

  Seasons to scrape:
    • saudi-professional-league-2022-2023
    • saudi-professional-league-2023-2024
    • saudi-professional-league-2024-2025
    • saudi-professional-league-2025-2026

  →  Starting headless Chrome …
  ✓  Browser ready.

══════════════════════════════════════════════════════════
  SEASON 1/4  ·  saudi-professional-league-2022-2023
══════════════════════════════════════════════════════════
  →  Detecting standings token for saudi-professional-league-2022-2023 …
  ✓  Token detected: CG4WPVLH  (from https://us.soccerway.com/saudi-arabia/saudi-professional-league-2022-2023/standings/CG4WPVLH/standings/overall/)

──────────────────────────────────────────────────────────
  1/8  ·  [2022-2023